# TakeMeter: Colab runner

**Runtime → Change runtime type → T4 GPU** before running anything.

This runs the whole pipeline (split → Groq baseline → fine-tune DistilBERT → evaluate) and fills in the README tables. Needs `data/takemeter_nba.csv` in the repo and a Colab secret named `GROQ_API_KEY`.

## 1. Get the repo
Option A: clone from GitHub (edit the URL). Option B: upload the repo as a zip.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/ai201-project3-takemeter.git"  # <- edit

import os
if not os.path.exists("takemeter"):
    !git clone $REPO_URL takemeter
%cd takemeter
!ls

In [ ]:
# Option B instead of the cell above: upload takemeter.zip
# from google.colab import files
# up = files.upload(); !unzip -q -o takemeter.zip; %cd takemeter

## 2. Setup

In [ ]:
!pip -q install gradio
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4!')

import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')  # never paste the key into a committed file

## 3. Baseline + fine-tune + evaluate
The baseline runs on the locked test set first and is cached in `outputs/baseline_predictions.csv`.

In [ ]:
!python train_eval.py --data data/takemeter_nba.csv

## 4. Fill the README tables from the real outputs

In [ ]:
!python scripts/make_report.py
from IPython.display import Image, display
display(Image('outputs/confusion_matrix.png'))

## 5. Download results to commit to GitHub
(`outputs/model/` is large and is git-ignored.)

In [ ]:
from google.colab import files
for f in ['README.md', 'outputs/evaluation_results.json', 'outputs/confusion_matrix.png',
          'outputs/confusion_matrix_baseline.png', 'outputs/test_predictions.csv',
          'outputs/baseline_predictions.csv', 'outputs/splits.csv']:
    if os.path.exists(f): files.download(f)

## 6. Demo (for the video)
Terminal-style classification, then the web UI with a public link.

In [ ]:
!python app.py "LETS GOOOOO WHAT A SHOT"
!python app.py "Luka is overrated. He's shooting 31% from three in the playoffs."

In [ ]:
!python app.py --share